In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [2]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)
train=pd.concat([pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/SDNFlow_Dataset_Normal.csv'),pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/SDNFlow_Dataset_attack.csv')])
tester=pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/modified_concatenated_InSDN_DatasetCSV.csv')

In [3]:
train=train.drop(columns=['flow_ID','eth_type','ipv4_src','ipv4_dst','ip_proto','src_port','dst_port','TnBPDstIP','TnBPSrcIP','TnPPDstIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP','Attack'])

In [4]:
tester = tester.drop(columns=[
    'Pkt Len Min',
    'Pkt Len Max',
    'Pkt Len Mean',
    'Pkt Len Std',
    'Pkt Len Var',
    'FIN Flag Cnt',
    'SYN Flag Cnt',
    'RST Flag Cnt',
    'PSH Flag Cnt',
    'ACK Flag Cnt',
    'URG Flag Cnt',
    'CWE Flag Count',
    'ECE Flag Cnt',
    'Down/Up Ratio'
])

In [5]:
tester = tester.drop(columns=[
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Tot',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Tot',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'Fwd Header Len',
    'Bwd Header Len'
])

In [6]:
tester = tester.drop(columns=[
    'Fwd Pkt Len Max',
    'Fwd Pkt Len Min',
    'Fwd Pkt Len Mean',
    'Fwd Pkt Len Std',
    'Bwd Pkt Len Max',
    'Bwd Pkt Len Min',
    'Bwd Pkt Len Mean',
    'Bwd Pkt Len Std'
])

In [7]:
tester=tester.drop(columns=['Flow ID','Timestamp','Flow Duration'])

tester['Tot Fwd Pkts']=tester['Tot Fwd Pkts']+tester['Tot Bwd Pkts']
tester=tester.rename(columns={'Tot Fwd Pkts':'tot_pkts'})

tester['TotLen Fwd Pkts']=tester['TotLen Fwd Pkts']+tester['TotLen Bwd Pkts']
tester=tester.rename(columns={'TotLen Fwd Pkts':'tot_len_pkts'})

tester=tester.drop(columns=['Tot Bwd Pkts','TotLen Bwd Pkts','Fwd Pkts/s','Bwd Pkts/s'])

In [8]:
tester=tester.rename(columns={'Flow Byts/s':'flow_byts_s','Flow Pkts/s':'flow_pkts_s','Pkt Size Avg':'pkt_size_average', 'Active Mean': 'active_mean',
    'Active Std': 'active_std',
    'Active Max': 'active_max',
    'Active Min': 'active_min',
    'Idle Mean': 'idle_mean',
    'Idle Std': 'idle_std',
    'Idle Max': 'idle_max',
    'Idle Min': 'idle_min'
})

In [9]:
train = train.drop(columns=[
    'fwd_header_len',
    'Byts_max_interval_2s',
    'Byts_min_interval_2s',
    'Byts_mean_interval_2s',
    'Byts_std_interval_2s',
    'Pkts_max_interval_2s',
    'Pkts_mean_interval_2s',
    'Pkts_min_interval_2s',
    'Pkts_std_interval_2s'
])

In [10]:
train=train.drop(columns='flow_duration')
train.head()
y=train['Category']
train.head()
X=train
y.head()
y=y.str.upper()
y.head()
y.value_counts()
y=y[~y.isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
y.value_counts()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.upper()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.strip()
tester['Label'].value_counts()

Label
DDOS          121942
PROBE          98129
NORMAL         68424
DOS            53616
BFA             1405
WEB-ATTACK       192
BOTNET           164
U2R               17
Name: count, dtype: int64

In [11]:
y=y.replace({
    'HTTP':'NORMAL',
    'PASSWORD-GUESSING':'BFA',
    'DNS':'BOTNET'
})
y.value_counts()
y.isnull().sum()
X.isnull().sum()
from sklearn.model_selection import train_test_split
X.shape
y.shape
y.head()
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
X.shape
y.shape

(659789,)

In [12]:
train['Category'].value_counts()
safe_train=train
train['Category']=train['Category'].str.strip()
train['Category']=train['Category'].str.upper()
train['Category'].value_counts()
train=train[~train['Category'].isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
train['Category'].value_counts()
y=train['Category']
y=le.fit_transform(y)
df=pd.DataFrame(y)
df.value_counts()

0
3    164974
5    151456
2    123807
0     74644
4     67288
1     53551
6     17787
7      6282
Name: count, dtype: int64

In [13]:
X=train.drop(columns='Category')
X.shape
y.shape
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [14]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
from imblearn.over_sampling import SMOTE;
smote=SMOTE();
Xoversampled,yoversampled=smote.fit_resample(X_train,y_train)

In [15]:
from xgboost import XGBClassifier


xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_train)),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42
)


estimators=[('xgb',xgb)]


from sklearn.pipeline import Pipeline


pipe=Pipeline(steps=estimators)

from skopt import BayesSearchCV


from skopt.space import Real,Categorical,Integer


search_space = {
    'max_depth': Integer(3, 10),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(400, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}

opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=42)


opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

eval_set = [
    (X_train, y_train)
]

opt.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_test, y_test)
    ],
    verbose=False
)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


BayesSearchCV(cv=3,
              estimator=XGBClassifier(base_score=None, booster=None,
                                      callbacks=None, colsample_bylevel=None,
                                      colsample_bynode=None,
                                      colsample_bytree=None, device=None,
                                      early_stopping_rounds=None,
                                      enable_categorical=False,
                                      eval_metric='mlogloss',
                                      feature_types=None, feature_weights=None,
                                      gamma=None, grow_policy=None,
                                      importance_type=None,
                                      interaction_constrai...
                             'max_depth': Integer(low=3, high=10, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=400, high=500, prior='uniform', transform='normalize'),
                             'reg_alpha': Real(low=0.0, high=5.0, prior='uniform', transform='normalize'),
                             'reg_lambda': Real(low=0.1, high=5.0, prior='uniform', transform='normalize'),
                             'subsample': Real(low=0.6, high=1.0, prior='uniform', transform='normalize')},
              verbose=1)

In [16]:
from sklearn.metrics import accuracy_score,classification_report

In [17]:
ypred=opt.predict(X_test)

In [18]:
accuracy_score(y_test,ypred)

0.7192818927234423

In [19]:
classification_report(y_test,ypred)

'              precision    recall  f1-score   support\n\n           0       0.95      0.27      0.42     14929\n           1       0.89      0.24      0.37     10710\n           2       0.93      0.91      0.92     24762\n           3       0.90      0.68      0.78     32995\n           4       0.94      0.78      0.85     13458\n           5       0.48      0.96      0.64     30291\n           6       0.90      0.76      0.82      3557\n           7       0.91      0.79      0.84      1256\n\n    accuracy                           0.72    131958\n   macro avg       0.86      0.67      0.71    131958\nweighted avg       0.82      0.72      0.71    131958\n'